# Extract MFCC Segment Datasets

This notebook creates train, validation and test datasets based on randomly sampled MFCC segments.

For each selected track, a fixed number of audio segments is sampled. For each segment, an MFCC matrix is extracted and stored as a model input.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()

for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import os
import glob
import pandas as pd
from tqdm import tqdm

from src.config import (
    PROCESSED_TRACKS_CSV,
    WAV_AUDIO_DIR,
    SPLIT_OUTPUT_DIR,
    SAMPLE_RATE,
    N_MFCC,
    SEGMENT_LENGTH_SECONDS,
    N_SEGMENTS_PER_TRACK,
    N_TRAIN_PER_GENRE,
    N_VAL_PER_GENRE,
    N_TEST_PER_GENRE,
    EXCLUDED_GENRES,
    USE_TOP_GENRES,
    TOP_N_GENRES
)

from src.preprocessing_functions import (
    fixed_split_per_genre_grouped,
    extract_random_mfcc_segments
)

In [ ]:
base_dir = str(WAV_AUDIO_DIR)
output_dir = str(SPLIT_OUTPUT_DIR)

LENGTH_IN_SECONDS = SEGMENT_LENGTH_SECONDS
N_SAMPLES = N_SEGMENTS_PER_TRACK

print("Base directory:", base_dir)
print("Output directory:", output_dir)
print("Sample rate:", SAMPLE_RATE)
print("MFCC coefficients:", N_MFCC)
print("Segment length:", LENGTH_IN_SECONDS)
print("Segments per track:", N_SAMPLES)

## Load and filter metadata

The metadata is filtered to tracks for which a corresponding WAV file is available.

In [ ]:
metadata = pd.read_csv(PROCESSED_TRACKS_CSV)[
    [
        "track_id",
        "track_genre_top"
    ]
]

available_files = [f for f in os.listdir(base_dir) if f.endswith(".wav")]
print(len(available_files), "WAV files found in directory.")

if any("_aug" in f for f in available_files):
    sep = "_"
else:
    sep = "."

print(f"Used separator: '{sep}'")

available_track_ids = set()

for f in available_files:
    try:
        track_id = str(int(f.split(sep)[0]))
        available_track_ids.add(track_id)
    except ValueError:
        print(f"Track_ID could not be extracted: {f}")

metadata_filtered = metadata[
    metadata["track_id"].astype(str).isin(available_track_ids)
]

print(f"Metadata before: {len(metadata)}")
print(f"Metadata after filtering on available track IDs: {len(metadata_filtered)}")

In [ ]:
metadata_filtered["track_genre_top"].value_counts(dropna=False)

## Create balanced train, validation and test splits

The dataset is split into training, validation and test sets using a fixed number of tracks per genre.

In [ ]:
splits, split_files = fixed_split_per_genre_grouped(
    metadata_filtered,
    base_dir,
    n_train=N_TRAIN_PER_GENRE,
    n_val=N_VAL_PER_GENRE,
    n_test=N_TEST_PER_GENRE,
    separator=sep,
    exclude_genres=EXCLUDED_GENRES,
    use_top_genres=USE_TOP_GENRES,
    top_n=TOP_N_GENRES
)

## Extract MFCC segment matrices

For each track, random audio segments are sampled and converted into MFCC matrices.  
The resulting datasets are stored as pickle files.

In [ ]:
base_dirs = {
    "train": base_dir,
    "val": base_dir,
    "test": base_dir
}

os.makedirs(output_dir, exist_ok=True)

for split_name, split_df in splits.items():
    print(f"\n[INFO] Processing split: {split_name} ({len(split_df)} tracks)")

    current_base_dir = base_dirs[split_name]
    data = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name}"):
        track_id = str(row["track_id"])
        genre = row["track_genre_top"]

        track_id_padded = str(int(track_id)).zfill(6)
        pattern = os.path.join(current_base_dir, f"{track_id_padded}*.wav")
        file_list = sorted(glob.glob(pattern))

        if not file_list:
            print(f"[WARN] No files found for Track {track_id} in {current_base_dir}")
            continue

        for file_path in file_list:
            file_name = os.path.basename(file_path)

            mfcc_segments = extract_random_mfcc_segments(
                file_path=file_path,
                length_in_seconds=LENGTH_IN_SECONDS,
                n_samples=N_SAMPLES,
                sr=SAMPLE_RATE,
                n_mfcc=N_MFCC
            )

            for segment_index, mfcc in enumerate(mfcc_segments):
                data.append(
                    {
                        "track_id": track_id,
                        "segment": segment_index,
                        "X": mfcc,
                        "split": split_name,
                        "genre": genre,
                        "file_name": file_name,
                        "is_augmented": "_aug" in file_name.lower()
                    }
                )

    if data:
        df_mfcc = pd.DataFrame(data)

        out_path = os.path.join(
            output_dir,
            f"{split_name}_large_mfcc_{N_SAMPLES}_{LENGTH_IN_SECONDS}s_{N_MFCC}mfcc_427_53_53_filtered_genres.pkl"
        )

        df_mfcc.to_pickle(out_path)

        print(f"[SAVED] {split_name}: {len(df_mfcc)} segments -> {out_path}")

    else:
        print(f"[WARN] No features calculated for split {split_name}!")

In [ ]:
for split_name in ["train", "val", "test"]:
    file_path = os.path.join(
        output_dir,
        f"{split_name}_large_mfcc_{N_SAMPLES}_{LENGTH_IN_SECONDS}s_{N_MFCC}mfcc_427_53_53_filtered_genres.pkl"
    )

    if os.path.exists(file_path):
        df = pd.read_pickle(file_path)
        print(split_name, df.shape)
        print(df["genre"].value_counts())
        print()
    else:
        print(f"{split_name}: file not found")